In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from tabulate import tabulate
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
from reportlab.lib.utils import ImageReader
from reportlab.platypus import Paragraph, Frame
from reportlab.lib.styles import getSampleStyleSheet
import os
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "dataoutputs"
IMAGES_DIR = DATA_DIR / "images"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def species_info(species):
    headers = {"User-Agent":""}
    url="https://waarnemingen.be/species/search/"
    wiki_url=f"https://en.wikipedia.org/wiki/{species}"
    search_parameters={"q":species,"species_group":16}
    response = requests.get(url,params=search_parameters)
    rarity_soup = BeautifulSoup(response.text,"html.parser")
    rarity_status_dutch = rarity_soup.find("span",class_="hidden-sm").get_text(strip=True)
    translation_api = "https://api.mymemory.translated.net/get"
    params = {
        "q": rarity_status_dutch,
        "langpair": "nl|en"
    }
    r = requests.get(translation_api, params=params, timeout=10)
    data = r.json()
    rarity_status_english = data["responseData"]["translatedText"]
    soup = BeautifulSoup(response.text,"html.parser")
    new_url = soup.find("a",href=re.compile(r"^/species/\d+/$"))
    if new_url is None:
        return "This species doesnot belong to Coleoptera,Thank you"
    species_url="https://waarnemingen.be"+new_url["href"]
    response2=requests.get(species_url)
    soup2=BeautifulSoup(response2.text,"html.parser")
    description_in_dutch = soup2.find("div",class_="app-content-section js-user-content").get_text(strip=True)
    response3=requests.get(wiki_url,headers=headers)
    soup3=BeautifulSoup(response3.text,"html.parser")
    div_container=soup3.find("div",class_="mw-content-ltr mw-parser-output")
    para=""
    for i in div_container.find_all("p"):
        if i.get_text(strip=True):
            para=i
            break
    for j in para.find_all("sup"):
        j.decompose()
    for k in para.find_all("style"):
        k.decompose()
    description_in_english=para.get_text()

    namur_url="https://waarnemingen.be"+new_url["href"]+"observations/?date_after=2024-12-25&date_before=2025-12-25&country_division=24&search=&user=&location=&sex=&month=&life_stage=&activity=&method="
    response4=requests.get(namur_url)
    soup4=BeautifulSoup(response4.text,"html.parser")
    table = soup4.find("table",class_="table table-bordered table-striped")
    lines = table.find("tbody").find_all("tr")
    data = []
    for i in lines:
        obs=i.find_all("td")
        if len(obs)<3:
            continue
        date = obs[0].get_text(strip=True)
        rawnumber = obs[1].get_text(" ", strip=True)
        match = re.search(r"\d+", rawnumber)
        number = match.group()
        location = obs[2].get_text(strip=True)
        data.append([date,number,location])
        if len(data)==10:
            break
        
    species_img = None
    img_tags = soup2.find_all("img")
    img_url = None
    for i in img_tags:
        src=i.get("src")
        if src.startswith("/media/photo/"):
            img_url="https://waarnemingen.be"+src
            break

    if img_url:
        response_img=requests.get(img_url)
        if response_img.status_code==200:
            species_img = IMAGES_DIR / f"{species}_img.jpg"
            with open(species_img,"wb") as f:
                for pic in response_img.iter_content(1024):
                    f.write(pic)
    
    full_species_informations = {
        "Name":species,
        "Latin Name":species,
        "Rarity Status(Dutch)":rarity_status_dutch,
        "Rarity Status(english)":rarity_status_english,
        "Description(Dutch)":description_in_dutch,
        "Description(English)":description_in_english,
        "Observation of the species in Namur": data,
        "image_path": str(species_img)
        
    }


    return full_species_informations

def draw_page_border(c, width, height, margin=1.5*cm):
    c.setLineWidth(1)
    c.rect(
        margin,
        margin,
        width - 2*margin,
        height - 2*margin
    )

def generate_species_pdf(
    species,
    latin_name,
    rarity_status_dutch,
    rarity_status_english,
    description_dutch,
    description_english,
    observations,
    image_path
):
    pdf_file = f"{species}_report.pdf"
    c = canvas.Canvas(pdf_file, pagesize=A4)
    width, height = A4

    margin = 1.5 * cm

    # ==========================
    # PAGE BORDER
    # ==========================
    c.setLineWidth(1)
    c.rect(
        margin,
        margin,
        width - 2 * margin,
        height - 2 * margin
    )

    styles = getSampleStyleSheet()
    normal = styles["Normal"]
    heading = styles["Heading2"]

    # ==========================
    # HEADER BLOCK
    # ==========================
    header_top = height - margin - 0.5 * cm
    text_x = margin + 0.5 * cm

    c.setFont("Helvetica-Bold", 16)
    c.drawString(text_x, header_top, species)

    c.setFont("Helvetica", 11)
    c.drawString(text_x, header_top - 1.0 * cm, f"Latin name: {latin_name}")
    c.drawString(
        text_x,
        header_top - 1.7 * cm,
        f"Rarity (NL): {rarity_status_dutch}"
    )
    c.drawString(
        text_x,
        header_top - 2.4 * cm,
        f"Rarity (EN): {rarity_status_english}"
    )

    # ==========================
    # IMAGE (FIXED BOX)
    # ==========================
    image_box = 4.5 * cm
    if image_path and os.path.exists(image_path):
        try:
            img = ImageReader(image_path)
            c.drawImage(
                img,
                width - margin - image_box,
                header_top - image_box,
                width=image_box,
                height=image_box,
                preserveAspectRatio=True,
                mask="auto"
            )
        except Exception:
            pass

    # ==========================
    # CONTENT FRAME (ONE PAGE)
    # ==========================
    header_height = 5.5 * cm  # real measured header
    frame = Frame(
        margin + 0.5 * cm,
        margin + 0.5 * cm,
        width - 2 * margin - 1 * cm,
        height - header_height - margin,
        showBoundary=0
    )

    story = []

    story.append(Paragraph("<b>Description (English)</b>", heading))
    story.append(Paragraph(description_english, normal))
    story.append(Paragraph("<br/>", normal))

    story.append(Paragraph("<b>Description (Dutch)</b>", heading))
    story.append(Paragraph(description_dutch, normal))
    story.append(Paragraph("<br/>", normal))

    story.append(Paragraph("<b>Recent observations in Namur</b>", heading))

    if observations:
        for date, number, location in observations:
            story.append(
                Paragraph(
                    f"<b>{date}</b> — {number} observed at {location}",
                    normal
                )
            )
    else:
        story.append(Paragraph("No observations available.", normal))

    frame.addFromList(story, c)

    c.save()
    print(f"PDF generated: {pdf_file}")
    return {
    "pdf_file": pdf_file,
    "species": species,
    "image_used": image_path,
    "observations_count": len(observations) if observations else 0
    }

species_info("Laccophilus minutus")

NameError: name 'species' is not defined

In [6]:
from bs4 import BeautifulSoup
import requests
import re
import folium
import time
import datetime
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from folium.plugins import MarkerCluster
import random
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
DATA_DIR = BASE_DIR / "dataoutputs"
MAP_DIR = DATA_DIR / "maps"
MAP_DIR.mkdir(parents=True, exist_ok=True)

def observations_map(day=None):
    if day is None:
        day = datetime.datetime.now().strftime("%Y-%m-%d")
    url = f"https://waarnemingen.be/fieldwork/observations/daylist/?date={day}&species_group=16&country_division=24&rarity=&search="
    response = requests.get(url)
    soup=BeautifulSoup(response.text,"html.parser")
    data_table = soup.find("table",class_="table table-bordered table-striped")
    if not data_table:
        print("opps..No observation found")
        return
    data_rows = data_table.find("tbody").find_all("tr")
    locator = Nominatim(user_agent="sadman")
    geocode = RateLimiter(locator.geocode)
    X = folium.Map(location=[50.4674,4.8718],zoom_start=9)
    cluster = MarkerCluster().add_to(X)
    color_for_species = {}

    for i in data_rows:
        data_col=i.find_all("td")
        if len(data_col)<5:
            continue
        species_namess=data_col[3]
        Common_name = species_namess.find("span", class_="species-common-name")
        scientific_name=species_namess.find("i",class_="species-scientific-name")
        if Common_name:
            name=Common_name.get_text(strip=True)
        elif scientific_name:
            name=scientific_name.get_text(strip=True)
        else:
            continue

        location = data_col[4]
        location_name=location.find_all("a")
        for i in location_name:
            raw_location=i.get_text(strip=True)
            clean_location=raw_location.split("(")[0].split(",")[0].strip()
            if name not in color_for_species:
                color_for_species[name] = "#{:06x}".format(random.randint(0, 0xFFFFFF))
            realLocation=geocode(clean_location+ ",Namur,Belgium")
            if not realLocation:
                continue
            folium.Marker(location=[realLocation.latitude,realLocation.longitude],popup=f"<b>{name}</b><br>{raw_location}</br>{day}",icon=folium.Icon(color=color_for_species[name],icon="smile",prefix="fa")).add_to(cluster)

    legend_html = """
        <div style="
            position: fixed;
            bottom: 50px;
            left: 50px;
            z-index: 9999;
            background-color: white;
            padding: 10px;
            border: 2px solid grey;
            font-size: 14px;
        ">
        <b>Species legend</b><br>
        """

    for species, color in color_for_species.items():
            legend_html += f"""
            <i style="color:{color}">●</i> {species}<br>
            """

    legend_html += "</div>"
    X.get_root().html.add_child(folium.Element(legend_html))
    mapfilename = MAP_DIR / f"{day}.html"
    X.save(mapfilename)
    return {
    "date": day,
    "map_file": str(mapfilename),
    "species_count": len(color_for_species),
    "species_colors": color_for_species,
    "map_object": X
    }

observations_map("2025-04-18")

/var/folders/76/z1mjvk1s7k36fwnyfkcvx1q40000gn/T/ipykernel_1804/2024064938.py:59: UserWarning: color argument of Icon should be one of: {'pink', 'purple', 'lightgray', 'cadetblue', 'blue', 'white', 'red', 'lightblue', 'gray', 'black', 'lightred', 'lightgreen', 'darkred', 'darkblue', 'green', 'orange', 'beige', 'darkpurple', 'darkgreen'}.
  folium.Marker(location=[realLocation.latitude,realLocation.longitude],popup=f"<b>{name}</b><br>{raw_location}</br>{day}",icon=folium.Icon(color=color_for_species[name],icon="smile",prefix="fa")).add_to(cluster)


{'date': '2025-04-18',
 'map_file': '/Users/mohammedsadmanuddin/python project/Take home project python/Project/dataoutputs/maps/2025-04-18.html',
 'species_count': 49,
 'species_colors': {'Gyrinus spec.': '#b3029f',
  'Gewone breedborst': '#24724d',
  'Echte breedborst': '#d2fc54',
  'Groene zandloopkever': '#5a9bf0',
  'Kortnekloopkever onbekend': '#b1ef73',
  'Bosspiegelloopkever': '#10f606',
  'Kielsprietloopkever onbekend': '#f04e12',
  'Rondhalszwartschild': '#8ad709',
  'Diksprietwaterroofkever': '#3d5a98',
  'Agabus paludosus': '#987178',
  'Gewone geelrand': '#6fac32',
  'Hydroglyphus geminus': '#a31144',
  'Hydroporus erythrocephalus': '#d83f31',
  'Hydroporus incognitus': '#24cea0',
  'Moeraswaterroofkevertje': '#79f43a',
  'Dwergwatertor': '#b476b2',
  'Hygrotus impressopunctatus': '#d64108',
  'Eironde watertor': '#859ccf',
  'Ilybius spec.': '#d9d8cf',
  'Laccophilus minutus': '#e89b21',
  'Grote spinnende watertor': '#0bd57e',
  'Harige kortschildkever': '#25baa0',
  'La